# Notebook overview: R8_5-CO_behaviorbased.ipynb

## What this notebook does
Builds behavior-informed evacuation/survival models for Colorado users. It merges network position, mobility behavior, distance, evacuation, and demographic features, fits predictive models, and generates user-level probabilities used in behavior-informed counterfactual sampling.

## Inputs used
- User-level feature table: user_level_feature_table_{revision}.csv
- Evacuation labels by user
- Home CBG and census demographic attributes
- Observed pre/post graph-derived centralities
- Mobility behavior variables such as entropy, exploration, repeat visitation, and travel distances

## Outputs created
- Behavior-informed prediction tables with yhat/probabilities
- Model diagnostic tables/plots
- Files used by R8_6/R8_10 for behavior-informed counterfactual graph construction
Behavior-informed counterfactual graphs are saved under: ../Data/stop_df_perday_CO/graphs_POI_weighted/boots_user_removal_weighted_yhat_count/{revision}/

**Data access warning.** The Cuebiq/Spectus mobility stop data used by this project are proprietary/restricted and are not included in this repository. Do not commit raw mobility files, user IDs, stop tables, home-location files, graph outputs containing sensitive identifiers, or large derived files unless your data-use agreement explicitly permits release. Public users must obtain data access independently and place files in the documented paths.

In [ ]:
pip install statsmodels 

In [ ]:
pip install seaborn

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import numpy as np
from scipy.sparse import lil_matrix
import json
from collections import defaultdict
import networkx as nx
import pickle
import os
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely import wkt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
import seaborn as sns

In [ ]:
print("Current working directory:", os.getcwd())
base = os.path.join("..", "Data", "stop_df_perday_CO")
pois_dir = os.path.join(base, "POIs")
geo_dir = os.path.join(base, "geography_CO")
stops_dir = os.path.join(base, "daily_agg_to_weekly_Stops")
graph_poi_dir = os.path.join(base, "graphs", "poi-user")
os.makedirs(graph_poi_dir, exist_ok=True)
week_dir = os.path.join(graph_poi_dir, "user_connections" )
os.makedirs(week_dir, exist_ok=True)
home_dir = os.path.join(base, "home/pre_disaster")
graph_dir = os.path.join(base, "graphs_POI_weighted")
os.makedirs(graph_dir, exist_ok=True)
revision = "R11"

boot_dir = os.path.join(graph_poi_dir, "boots_user_removal", revision)
cf_out_dir = os.path.join(graph_poi_dir, "counterfactual_post", revision)

boot_dir_w = os.path.join(graph_poi_dir, "boots_user_removal_weighted_yhat", revision)
os.makedirs(boot_dir_w, exist_ok=True)

boot_dir_w_count = os.path.join(graph_poi_dir, "boots_user_removal_weighted_yhat_count", revision)
os.makedirs(boot_dir_w_count, exist_ok=True)

os.makedirs(cf_out_dir, exist_ok=True)

survivor_dir = os.path.join(graph_poi_dir, "boots_user_survivors", revision)
os.makedirs(survivor_dir, exist_ok=True)

# y-hat computation

In [ ]:
out_path = os.path.join(graph_dir, f"user_level_feature_table_{revision}.csv")
user_feature = pd.read_csv(out_path)

In [ ]:
user_feature.columns

In [ ]:
user_feature["evacuation"]

In [ ]:
scale_cols = [
    "total_population", 
    "median_age",
    #"white_percent", 
    "black_percent",
    "median_income_log", 
    "median_rent_log", 
    "fire_dist",
    #"raw_degree_pre_mean",
    'strength_pre_mean',
    'closeness_centrality_pre_mean', 
    'clustering_coefficient_pre_mean',
    "H_weighted_pre",
   #"H_poi_pre", 
    "poi_explore_rate_pre", 
    "avg_dist_km",
    #"n_pois_interaction"
]

In [ ]:
user_feature.dropna(subset=scale_cols + ["evacuation"], inplace=True)
user_feature.shape

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler = MinMaxScaler().fit(user_feature[scale_cols])
user_feature[scale_cols] = scaler.transform(user_feature[scale_cols])

In [ ]:
formula_1 = ( "evacuation ~  " + " + ".join(scale_cols))
formula_2 = ( "random_evacuation_prob ~  " + " + ".join(scale_cols))

In [ ]:
m_1   = smf.logit(formula_1,   data=user_feature).fit(cov_type="HC3")
m_2   = smf.ols(formula_2,   data=user_feature).fit(cov_type="HC3")

In [ ]:
# m_1.summary()

In [ ]:
# m_2.summary()

In [ ]:
user_feature["yhat_user_evac"] = m_1.predict(user_feature)
user_feature["yhat_user_removal"] = m_2.predict(user_feature)

### Evacuation propensity model (Logit)

Let \(E_i \in \{0,1\}\) indicate whether user \(i\) evacuated \((E_i=1)\).

We estimate a logistic regression:

$
\Pr(E_i = 1) \;=\; \text{logit}^{-1}\!\Big(
\alpha
\;+\;\boldsymbol{\beta}^{\top}\mathbf{C}_i^{\text{pre}}
\;+\;\gamma\,D_i
\;+\;\boldsymbol{\delta}^{\top}\mathbf{SDM}_i
\;+\;\eta\,X_i^{\text{pre}}
\Big)
$


The fitted values \$\hat{p}_i = \Pr(E_i=1)$) are used as **propensity scores** (predicted evacuation probabilities) for behavior-informed counterfactual sampling.

% Evacuation-controlled random-removal null (single-equation form)
$G^{\mathrm{rand}}_{\,\tau}
\;=\;
G_{\mathrm{pre}}\!\left[
\,V_{\mathrm{pre}}\setminus
\bigcup_{g\in\mathcal{G}}
R_{g,\tau}
\right],
\qquad
R_{g,\tau}\sim \mathrm{Unif}\Big\{
S\subseteq V_g:\ |S|=\Delta I_{g,\tau}
\Big\}.$

($V_g=\{i\in V_{\mathrm{pre}}: \mathrm{CBG}(i)=g\}$, and $\Delta I_{g,\tau}$) is the observed number of evacuees in $CBG_g$ over window $\tau$.

empirical survival probability across R random runs:
	•	Let $S_i^{(r)} \in \{0,1\}$ indicate whether user i is present/survives in run r.
	•	Let $r_i = \sum_{r=1}^{R} S_i^{(r)} $ be n_random_runs_present.

Then the correct math is:

$\hat{p}^{\text{rand}}_{\text{survive},i}
= \frac{r_i}{R}
= \frac{1}{R}\sum_{r=1}^{R} S_i^{(r)},
\qquad
\hat{p}^{\text{rand}}_{\text{evac},i}
= 1-\hat{p}^{\text{rand}}_{\text{survive},i}.$

$p^{\mathrm{rand}}_i = \frac{m_{g(i)}}{N_{g(i)}}
\quad\text{with}\quad N_{g}=|V_g|$


$p^{\mathrm{rand}}_i \equiv \Pr_{\mathrm{null}}(Z_i=1)
= \frac{m_{g(i)}}{|V_{g(i)}|}
\approx \frac{1}{R}\sum_{r=1}^{R} Z_i^{(r)}.
\]$

In [ ]:
user_feature.shape

In [ ]:
def coef_plot_logit(
    m,
    var_pretty=None,
    *,
    group_order=("centrality", "behavior", "distance", "demography"),
    group_map=None,
    drop_intercept=True,
    title="Logit coefficients (95% CI)",
    xlabel="log-odds",
    figsize=(4, 6),
    alpha_sig=0.05,
    tick_fontsize=16,
    label_fontsize=16,
    title_fontsize=18,
    marker_size=55,
    ci_lw=1.8,
):
    var_pretty = {} if var_pretty is None else var_pretty

    default_group = {
        "raw_degree_pre_mean": "centrality",
        "strength_pre_mean": "centrality",
        "closeness_centrality_pre_mean": "centrality",
        "clustering_coefficient_pre_mean": "centrality",
        "poi_explore_rate_pre": "behavior",
        "H_weighted_pre": "behavior",
        "H_poi_pre": "behavior",
        "fire_dist": "distance",
        "avg_dist_km": "distance",
        "total_population": "demography",
        "median_age": "demography",
        "white_percent": "demography",
        "black_percent": "demography",
        "median_income_log": "demography",
        "median_rent_log": "demography",
    }
    if group_map is not None:
        default_group.update(group_map)

    group_rank = {g: i for i, g in enumerate(group_order)}

    group_colors = {
        "centrality": "black",  # blue
        "behavior":   "black",  # orange
        "distance":   "black",
        "demography": "black",  # green
        "other":      "black",
    }

    coefs = m_1.params.copy()
    ci = m_1.conf_int().copy()
    ci.columns = ["ci_low", "ci_high"]

    dfp = pd.DataFrame({
        "var": coefs.index.astype(str),
        "coef": coefs.values,
        "ci_low": ci.loc[coefs.index, "ci_low"].values,
        "ci_high": ci.loc[coefs.index, "ci_high"].values,
        "pval": m_1.pvalues.loc[coefs.index].values,
    })

    if drop_intercept:
        dfp = dfp[~dfp["var"].isin(["Intercept", "const"])].copy()

    dfp["var_pretty"] = dfp["var"].map(var_pretty).fillna(dfp["var"])
    dfp["group"] = dfp["var"].map(default_group).fillna("other")
    dfp["group_rank"] = dfp["group"].map(group_rank).fillna(99).astype(int)

    dfp["abs_coef"] = dfp["coef"].abs()
    dfp = dfp.sort_values(["group_rank", "abs_coef"], ascending=[True, True]).reset_index(drop=True)

    dfp["is_sig"] = dfp["pval"] < float(alpha_sig)

    y = np.arange(len(dfp))

    fig, ax = plt.subplots(figsize=figsize)

    # CI lines: grey if significant, lightgrey if not
    ci_colors = np.where(dfp["is_sig"], "grey", "lightgrey")
    ax.hlines(y, dfp["ci_low"], dfp["ci_high"], color=ci_colors, linewidth=ci_lw, alpha=0.9, zorder=1)

    # Points: group color if significant; otherwise lightgrey
    point_colors = [
        (group_colors.get(g, "grey") if sig else "lightgrey")
        for g, sig in zip(dfp["group"].tolist(), dfp["is_sig"].tolist())
    ]
    ax.scatter(dfp["coef"], y, s=marker_size, color=point_colors, zorder=3)

    ax.axvline(0, color="black", linestyle="--", linewidth=1)

    ax.set_yticks(y)
    ax.set_yticklabels(dfp["var_pretty"], fontsize=tick_fontsize)
    ax.set_xlabel(xlabel, fontsize=label_fontsize)
    ax.set_title(title, fontsize=title_fontsize)
    ax.tick_params(axis="x", labelsize=tick_fontsize)

    # faint separators between groups
    cuts = dfp["group_rank"].ne(dfp["group_rank"].shift()).to_numpy()
    cut_idx = np.where(cuts)[0]
    for i in cut_idx[1:]:
        ax.axhline(i - 0.5, color="black", linewidth=0.6, alpha=0.15)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    plt.tight_layout()
    plt.show()


var_pretty = {
    "total_population": "Total Pop",
    "median_age": "Median age",
    "black_percent": "Black (%)",
    "median_income_log": "Median income",
    "median_rent_log": "Median rent",
    "fire_dist": "Disaster distance",
    "raw_degree_pre_mean": "Pre degree",
    "strength_pre_mean": "Pre strength",
    "closeness_centrality_pre_mean": "Pre closeness",
    "clustering_coefficient_pre_mean": "Pre clustering",
    "poi_explore_rate_pre": "POI explore rate",
    "H_weighted_pre": "Category Entropy",
    "avg_dist_km": "Distance traveled",
}

title_formula = (
    r"$\Pr(E_i = 1) =\mathrm{logit}\left("
     r"\alpha"
    r"+\beta F_i"
    r"\beta_{1} C_{i}"
    r"+\beta_{2} D_i"
    r"+\beta_{3} SDM_i"
    r"+\beta_{4} X_{i}"
    r"\right)$"
)

coef_plot_logit(m_1, var_pretty=var_pretty, title= "$Pr(E_i = 1)$")

In [ ]:
def coef_plot_logit(
    m,
    var_pretty=None,
    *,
    group_order=("centrality", "behavior", "distance", "demography"),
    group_map=None,
    drop_intercept=True,
    title="Logit coefficients (95% CI)",
    xlabel="Coefficient (log-odds)",
    figsize=(5, 5),
    alpha_sig=0.05,
    tick_fontsize=14,
    label_fontsize=14,
    title_fontsize=16,
    marker_size=55,
    ci_lw=1.8,
):
    var_pretty = {} if var_pretty is None else var_pretty

    default_group = {
        "raw_degree_pre_mean": "centrality",
        "strength_pre_mean": "centrality",
        "closeness_centrality_pre_mean": "centrality",
        "clustering_coefficient_pre_mean": "centrality",
        "poi_explore_rate_pre": "behavior",
        "H_weighted_pre": "behavior",
        "H_poi_pre": "behavior",
        "fire_dist": "distance",
        "avg_dist_km": "distance",
        "total_population": "demography",
        "median_age": "demography",
        "white_percent": "demography",
        "black_percent": "demography",
        "median_income_log": "demography",
        "median_rent_log": "demography",
    }
    if group_map is not None:
        default_group.update(group_map)

    group_rank = {g: i for i, g in enumerate(group_order)}

    group_colors = {
        "centrality": "#1f77b4",  # blue
        "behavior":   "#ff7f0e",  # orange
        "distance":   "black",
        "demography": "#2ca02c",  # green
        "other":      "grey",
    }

    coefs = m_2.params.copy()
    ci = m_2.conf_int().copy()
    ci.columns = ["ci_low", "ci_high"]

    dfp = pd.DataFrame({
        "var": coefs.index.astype(str),
        "coef": coefs.values,
        "ci_low": ci.loc[coefs.index, "ci_low"].values,
        "ci_high": ci.loc[coefs.index, "ci_high"].values,
        "pval": m_2.pvalues.loc[coefs.index].values,
    })

    if drop_intercept:
        dfp = dfp[~dfp["var"].isin(["Intercept", "const"])].copy()

    dfp["var_pretty"] = dfp["var"].map(var_pretty).fillna(dfp["var"])
    dfp["group"] = dfp["var"].map(default_group).fillna("other")
    dfp["group_rank"] = dfp["group"].map(group_rank).fillna(99).astype(int)

    dfp["abs_coef"] = dfp["coef"].abs()
    dfp = dfp.sort_values(["group_rank", "abs_coef"], ascending=[True, True]).reset_index(drop=True)

    dfp["is_sig"] = dfp["pval"] < float(alpha_sig)

    y = np.arange(len(dfp))

    fig, ax = plt.subplots(figsize=figsize)

    # CI lines: grey if significant, lightgrey if not
    ci_colors = np.where(dfp["is_sig"], "grey", "lightgrey")
    ax.hlines(y, dfp["ci_low"], dfp["ci_high"], color=ci_colors, linewidth=ci_lw, alpha=0.9, zorder=1)

    # Points: group color if significant; otherwise lightgrey
    point_colors = [
        (group_colors.get(g, "grey") if sig else "lightgrey")
        for g, sig in zip(dfp["group"].tolist(), dfp["is_sig"].tolist())
    ]
    ax.scatter(dfp["coef"], y, s=marker_size, color=point_colors, zorder=3)

    ax.axvline(0, color="black", linestyle="--", linewidth=1)

    ax.set_yticks(y)
    ax.set_yticklabels(dfp["var_pretty"], fontsize=tick_fontsize)
    ax.set_xlabel(xlabel, fontsize=label_fontsize)
    ax.set_title(title, fontsize=title_fontsize)
    ax.tick_params(axis="x", labelsize=tick_fontsize)

    # faint separators between groups
    cuts = dfp["group_rank"].ne(dfp["group_rank"].shift()).to_numpy()
    cut_idx = np.where(cuts)[0]
    for i in cut_idx[1:]:
        ax.axhline(i - 0.5, color="black", linewidth=0.6, alpha=0.15)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    plt.tight_layout()
    plt.show()


var_pretty = {
    "total_population": "Total Pop",
    "median_age": "Median age",
    "black_percent": "Black (%)",
    "median_income_log": "Median income",
    "median_rent_log": "Median rent",
    "fire_dist": "Disaster distance",
    "raw_degree_pre_mean": "Pre degree",
    "strength_pre_mean": "Pre strength",
    "closeness_centrality_pre_mean": "Pre closeness",
    "clustering_coefficient_pre_mean": "Pre clustering",
    "poi_explore_rate_pre": "POI explore rate",
    "H_weighted_pre": "Category Entropy",
    "avg_dist_km": "Distance traveled",
}

title_formula = (
    r"$\Pr(E_i = 1)=\mathrm{logit}\left("
    r"\alpha"
    r"+\beta_{1} C_{i}"
    r"+\beta_{2} D_i"
    r"+\beta_{3} SDM_i"
    r"+\beta_{4} X_{i}"
    r"\right)$"
)

coef_plot_logit(m_2, var_pretty=var_pretty, title=title_formula)

In [ ]:
def coef_plot_two_models(
    m1,
    m2,
    var_pretty=None,
    *,
    label1="m1",
    label2="m2",
    marker1="o",
    marker2="s",
    group_order=("centrality", "behavior", "distance", "demography"),
    group_map=None,
    drop_intercept=True,
    title="Coefficients (95% CI)",
    xlabel="Coefficient",
    figsize=(5, 6.5),
    alpha_sig=0.05,
    tick_fontsize=16,
    label_fontsize=16,
    title_fontsize=18,
    marker_size=45,
    ci_lw=1.8,
    dodge=0.18,          # vertical dodge between models
):
    var_pretty = {} if var_pretty is None else var_pretty

    default_group = {
        "raw_degree_pre_mean": "centrality",
        "strength_pre_mean": "centrality",
        "closeness_centrality_pre_mean": "centrality",
        "clustering_coefficient_pre_mean": "centrality",
        "poi_explore_rate_pre": "behavior",
        "H_weighted_pre": "behavior",
        "H_poi_pre": "behavior",
        "fire_dist": "distance",
        "avg_dist_km": "distance",
        "total_population": "demography",
        "median_age": "demography",
        "white_percent": "demography",
        "black_percent": "demography",
        "median_income_log": "demography",
        "median_rent_log": "demography",
    }
    if group_map is not None:
        default_group.update(group_map)

    group_rank = {g: i for i, g in enumerate(group_order)}

    group_colors = {
        "centrality": "#1f77b4",  # blue
        "behavior":   "#ff7f0e",  # orange
        "distance":   "black",
        "demography": "#2ca02c",  # green
        "other":      "grey",
    }

    def tidy(model, model_label):
        coefs = model.params.copy()
        ci = model.conf_int().copy()
        ci.columns = ["ci_low", "ci_high"]
        pvals = model.pvalues.loc[coefs.index].copy()

        out = pd.DataFrame({
            "var": coefs.index.astype(str),
            "coef": coefs.values,
            "ci_low": ci.loc[coefs.index, "ci_low"].values,
            "ci_high": ci.loc[coefs.index, "ci_high"].values,
            "pval": pvals.values,
            "model": model_label,
        })

        if drop_intercept:
            out = out[~out["var"].isin(["Intercept", "const"])].copy()

        out["var_pretty"] = out["var"].map(var_pretty).fillna(out["var"])
        out["group"] = out["var"].map(default_group).fillna("other")
        out["group_rank"] = out["group"].map(group_rank).fillna(99).astype(int)
        out["abs_coef"] = out["coef"].abs()
        out["is_sig"] = out["pval"] < float(alpha_sig)
        return out

    d1 = tidy(m1, label1)
    d2 = tidy(m2, label2)

    # union of variables (keep order by group then avg |coef|)
    d_all = pd.concat([d1, d2], ignore_index=True)
    order_df = (
        d_all.groupby(["var", "var_pretty", "group", "group_rank"], as_index=False)
             .agg(abs_mean=("abs_coef", "mean"))
             .sort_values(["group_rank", "abs_mean"], ascending=[True, True])
    )
    ordered_vars = order_df["var"].tolist()

    # map var -> y base index
    y_base_map = {v: i for i, v in enumerate(ordered_vars)}

    d1["_y"] = d1["var"].map(y_base_map).astype(float) - dodge
    d2["_y"] = d2["var"].map(y_base_map).astype(float) + dodge

    fig, ax = plt.subplots(figsize=figsize)

    # CI colors (sig grey / nonsig lightgrey)
    for d, mk in [(d1, marker1), (d2, marker2)]:
        ci_colors = np.where(d["is_sig"], "grey", "lightgrey")
        ax.hlines(d["_y"], d["ci_low"], d["ci_high"], color=ci_colors, linewidth=ci_lw, alpha=0.9, zorder=1)

        pt_colors = [
            (group_colors.get(g, "grey") if sig else "lightgrey")
            for g, sig in zip(d["group"].tolist(), d["is_sig"].tolist())
        ]
        ax.scatter(d["coef"], d["_y"], s=marker_size, color=pt_colors, marker=mk, zorder=3)

    ax.axvline(0, color="black", linestyle="--", linewidth=1)

    # y ticks at base positions
    y_ticks = np.arange(len(ordered_vars))
    y_labels = [order_df.set_index("var").loc[v, "var_pretty"] for v in ordered_vars]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels, fontsize=tick_fontsize)

    ax.set_xlabel(xlabel, fontsize=label_fontsize)
    ax.set_title(title, fontsize=title_fontsize)
    ax.tick_params(axis="x", labelsize=tick_fontsize)

    # faint separators between groups
    grp_seq = order_df["group_rank"].to_numpy()
    cut_idx = np.where(grp_seq != np.r_[grp_seq[0], grp_seq[:-1]])[0]
    for i in cut_idx[1:]:
        ax.axhline(i - 0.5, color="black", linewidth=0.6, alpha=0.15)

    # minimal model marker key (no full legend)
    ax.scatter([], [], marker=marker1, color="black", label=label1)
    ax.scatter([], [], marker=marker2, color="black", label=label2)
    #ax.legend(frameon=False, fontsize=12, loc="lower right")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    plt.tight_layout()
    plt.show()


# ---- run ----
coef_plot_two_models(
    m_1, m_2,
    var_pretty=var_pretty,
    label1="Observed evac (Logit)",
    label2="Random evac prob (OLS)",
    marker1="o",
    marker2="s",
    #title=title_formula,
    xlabel="Coefficient",
)

In [ ]:
out_path = os.path.join(graph_dir, f"user_level_feature_table_yhat_{revision}_counts.csv")
user_feature.to_csv(out_path, index=False)
print("Saved:", out_path)

In [ ]:
user_feature

# Behavior informed randomization
higher y-hat users are more likely to be removed, but total removals per CBG are still fixed.

## Load pre and post graphs

In [ ]:
oct_path = os.path.join(graph_dir, f"PDM_Oct2021_graph_POI_weighted_{revision}.pkl")
nov_path = os.path.join(graph_dir, f"PDM_Nov2021_graph_POI_weighted_{revision}.pkl")
jan_remain_path = os.path.join(graph_dir, f"Jan2022_remaining_pre_users_graph_POI_weighted_{revision}.pkl")
feb_remain_path = os.path.join(graph_dir, f"Feb2022_remaining_pre_users_graph_POI_weighted_{revision}.pkl")

In [ ]:
with open(oct_path, "rb") as f:
    G_oct = pickle.load(f)
with open(nov_path, "rb") as f:
    G_nov = pickle.load(f)
with open(jan_remain_path, "rb") as f:
    G_jan = pickle.load(f)
with open(feb_remain_path, "rb") as f:
    G_feb = pickle.load(f)

In [ ]:
pre_union = set(map(str, G_oct.nodes())) | set(map(str, G_nov.nodes()))
jan_nodes = set(map(str, G_jan.nodes()))
feb_nodes = set(map(str, G_feb.nodes()))

post_union = jan_nodes | feb_nodes

print("Pre union (Oct ∪ Nov):", len(pre_union))
print("Jan remain:", len(jan_nodes))
print("Feb remain:", len(feb_nodes))
print("Post union (Jan ∪ Feb):", len(post_union))

print("\nChecks (should be 0):")
print("Jan not in pre_union:", len(jan_nodes - pre_union))
print("Feb not in pre_union:", len(feb_nodes - pre_union))
print("Post union not in pre_union:", len(post_union - pre_union))

print("\nObserved survivors among pre_union (Jan ∪ Feb):", len(pre_union & post_union))

In [ ]:
out_path = os.path.join(graph_dir, f"user_level_feature_table_yhat_{revision}_counts.csv")
user_feature = pd.read_csv(out_path)
user_feature.columns

In [ ]:
user_feature

## Random removal with $y^ˆ$ probability

In [ ]:
# this uses user feature dataset to build random - ignore this - use graph directly
user_feature["user"] = user_feature["user"].astype(str)
user_feature["pre_disaster_fips"] = user_feature["pre_disaster_fips"].astype(str)

user_feature["yhat_user_evac"] = pd.to_numeric(user_feature["yhat_user_evac"], errors="coerce")
user_feature["evacuation"] = pd.to_numeric(user_feature["evacuation"], errors="coerce")

ue = user_feature

post_union = set(map(str, post_union))
U = set(ue["user"].astype(str))
survived_obs_U  = set(post_union) & U
evacuated_obs_U = U - survived_obs_U

print("Universe |U|:", len(U))
print("Observed survivors in U:", len(survived_obs_U))
print("Observed evacuees in U:", len(evacuated_obs_U))

# pool of all users in each CBG
pool_by_cbg = (
    ue.groupby("pre_disaster_fips")["user"]
    .apply(lambda s: list(pd.unique(s.astype(str))))
    .to_dict()
)

# observed # evacuees per CBG = target removals
# L_by_cbg = (
#     ue.loc[ue["evacuation"] == 1]
#     .groupby("pre_disaster_fips")["user"]
#     .nunique()
# )

L_by_cbg = (
    ue.loc[ue["user"].astype(str).isin(evacuated_obs_U)]
    .groupby("pre_disaster_fips")["user"]
    .nunique()
)

# optional diagnostic
T_by_cbg = ue.groupby("pre_disaster_fips")["user"].nunique()

print("Total target removals (sum L_g):", int(L_by_cbg.sum()))
print("Total pre users (|U|):", len(U))
print("Implied survivors:", len(U) - int(L_by_cbg.sum()))

In [ ]:
def draw_weighted_random_evacuations(pool_by_cbg, L_by_cbg, user_yhat_map, rng):
    """
    Remove exactly L_g users per CBG, weighted by yhat_user_evac.
    """
    removed_set = set()
    removed_by_cbg = {}

    for cbg, users in pool_by_cbg.items():
        users = np.array(users, dtype=object)
        Lg = int(L_by_cbg.get(cbg, 0))

        if Lg <= 0:
            removed_by_cbg[cbg] = []
            continue

        # weights from yhat
        w = np.array([user_yhat_map[u] for u in users], dtype=float)

        # normalize within CBG
        p = w / w.sum()

        # weighted sample without replacement
        picked = rng.choice(users, size=Lg, replace=False, p=p)

        removed_by_cbg[cbg] = picked.tolist()
        removed_set.update(picked.tolist())

    return removed_set, removed_by_cbg

In [ ]:
def run_user_randomization_weighted(ue, pool_by_cbg, L_by_cbg, n_runs=500, seed=42, ci=(2.5, 97.5)):
    rng = np.random.default_rng(seed)

    users_all = ue["user"].astype(str).unique()
    user_to_idx = {u: i for i, u in enumerate(users_all)}

    # user -> yhat map
    user_yhat_map = (
        ue[["user", "yhat_user_evac"]]
        .drop_duplicates("user")
        .assign(user=lambda d: d["user"].astype(str))
        .set_index("user")["yhat_user_evac"]
        .to_dict()
    )

    removed_mat = np.zeros((n_runs, len(users_all)), dtype=np.uint8)

    cbg_checks = []
    target_total = int(L_by_cbg.sum())

    for r in range(n_runs):
        removed_set, removed_by_cbg = draw_weighted_random_evacuations(
            pool_by_cbg=pool_by_cbg,
            L_by_cbg=L_by_cbg,
            user_yhat_map=user_yhat_map,
            rng=rng
        )

        idx = [user_to_idx[u] for u in removed_set]
        removed_mat[r, idx] = 1

        cbg_checks.append({
            "run": r,
            "removed_total": int(removed_mat[r].sum()),
            "target_total": target_total
        })

    # per-user removal probability
    p_hat = removed_mat.mean(axis=0)
    lo = np.percentile(removed_mat, ci[0], axis=0)
    hi = np.percentile(removed_mat, ci[1], axis=0)

    user_ci_weighted = pd.DataFrame({
        "user": users_all,
        "p_removed_mean_weighted": p_hat,
        f"p_removed_p{ci[0]}_weighted": lo,
        f"p_removed_p{ci[1]}_weighted": hi
    })
    user_ci_weighted["p_survival_mean_weighted"] = 1 - user_ci_weighted["p_removed_mean_weighted"]

    cbg_checks = pd.DataFrame(cbg_checks)

    return removed_mat, user_ci_weighted, cbg_checks

In [ ]:
removed_mat_w, user_ci_weighted, cbg_checks_w = run_user_randomization_weighted(
    ue=ue,
    pool_by_cbg=pool_by_cbg,
    L_by_cbg=L_by_cbg,
    n_runs=500,
    seed=42
)

print(cbg_checks_w.head())
print(user_ci_weighted.sort_values("p_removed_mean_weighted", ascending=False).head(10))

In [ ]:
# quick check: any CBG with zero total yhat?
cbg_weight_sum = ue.groupby("pre_disaster_fips")["yhat_user_evac"].sum()
print((cbg_weight_sum == 0).sum(), "CBGs have zero total yhat")

In [ ]:
users_all = ue["user"].astype(str).unique()
users_all = np.array(users_all, dtype=object)

for r in range(removed_mat_w.shape[0]):
    removed_users = users_all[np.where(removed_mat_w[r] == 1)[0]].tolist()
    survivors = list(set(users_all) - set(removed_users))

    with open(os.path.join(boot_dir_w_count, f"removed_users_run{r:04d}.json"), "w") as f:
        json.dump(removed_users, f)

    with open(os.path.join(boot_dir_w_count, f"survivors_run{r:04d}.json"), "w") as f:
        json.dump(survivors, f)

## Comute complementary survivors to above random removals

In [ ]:
# Actual survivors from observed data
actual_survivors = set(
    ue.loc[ue["evacuation"] == 0, "user"].astype(str)
)

actual_survivor_count = len(actual_survivors)
print("Actual survivors:", actual_survivor_count)

users_all = ue["user"].astype(str).unique()
users_all = np.array(users_all, dtype=object)

survivor_counts = []

# use your weighted removed matrix here
for r in range(removed_mat_w.shape[0]):
    removed_idx = np.where(removed_mat_w[r] == 1)[0]
    removed_set = set(users_all[removed_idx])

    survivors_set = set(users_all) - removed_set
    survivor_counts.append(len(survivors_set))

    with open(os.path.join(boot_dir_w_count, f"survivors_run{r:04d}.json"), "w") as f:
        json.dump(list(survivors_set), f)

print("Saved weighted survivor sets.")
print("Actual survivor count:", actual_survivor_count)
print("Mean simulated survivor count:", np.mean(survivor_counts))
print("Min simulated survivor count:", np.min(survivor_counts))
print("Max simulated survivor count:", np.max(survivor_counts))

In [ ]:
boot_dir_w_count

## Building behavior informed random graphs

## Sub graph with random survivors

In [ ]:
survivor_files = sorted(
    [f for f in os.listdir(boot_dir_w) if f.startswith("survivors_run") and f.endswith(".json")]
)

cf_metrics = []

for f in survivor_files:
    run = int(f.split("run")[1].split(".")[0])

    # load survivors for this run
    with open(os.path.join(boot_dir_w_count, f), "r") as fp:
        survivors = set(map(int, json.load(fp)))   # match G_pre node type

    # keep only nodes that exist in G_pre (safe guard)
    survivors = survivors.intersection(G_oct.nodes)

    # induced counterfactual graph
    G_cf = G_oct.subgraph(survivors).copy()

    # Save graph in same directory (boot_dir_w)
    graph_path = os.path.join(
        boot_dir_w_count,
        f"CF_graph_run{run:04d}_Oct2021_{revision}.pkl"
    )
    with open(graph_path, "wb") as gf:
        pickle.dump(G_cf, gf)


    # Compute metrics
    nodes = G_cf.number_of_nodes()
    edges = G_cf.number_of_edges()

    mean_strength = (
        sum(dict(G_cf.degree(weight="weight")).values()) / nodes
        if nodes > 0 else 0.0
    )

    clustering = (
        nx.average_clustering(G_cf, weight="weight")
        if nodes > 1 else 0.0
    )

    cf_metrics.append({
        "run": run,
        "nodes": nodes,
        "edges": edges,
        "mean_strength": mean_strength,
        "clustering": clustering,
        "type": "random_post_yhat"
    })

    print(f"Run {run:04d} | Nodes={nodes} | Edges={edges} | Mean strength={mean_strength:.3f}")



## save metrics

In [ ]:
cf_metrics_df = pd.DataFrame(cf_metrics)

metrics_path = os.path.join(
    boot_dir_w_count,
    f"CF_network_metrics_Oct2021_yhat_{revision}_count.csv"
)

cf_metrics_df.to_csv(metrics_path, index=False)

cf_metrics_df.head()

## Load and combine all graph Metrics

In [ ]:
random_network_path = os.path.join(
    survivor_dir,
    f"CF_network_metrics_Oct2021_{revision}.csv"
)

random_network_df = pd.read_csv(random_network_path)
random_network_df.head()

In [ ]:
random_structural = random_network_df[[
    "nodes",
    "edges",
    "mean_strength",
    "run"
]].copy()

random_structural["month"] = "random_post"
random_structural["phase"] = "random"

In [ ]:
yhat_network_path = os.path.join(
    boot_dir_w_count,
    f"CF_network_metrics_Oct2021_yhat_{revision}_count.csv"
)

yhat_network_df = pd.read_csv(yhat_network_path)
yhat_network_df.head()

In [ ]:
yhat_structural = yhat_network_df[[
    "nodes",
    "edges",
    "mean_strength",
    "run"
]].copy()

yhat_structural["month"] = "yhat_post"
yhat_structural["phase"] = "yhat"

In [ ]:
graph_dir = "../Data/stop_df_perday_CO/graphs_POI_weighted"
network_path = os.path.join(
    graph_dir,
    "network_centrality_POI_weighted_all_revisions.csv"
)

network_df = pd.read_csv(network_path)
# Filter R11 only
r11_df = network_df[network_df["revision"] == "R11"].copy()

# Keep only required columns
r11_structural = r11_df[[
    "month",
    "phase",
    "nodes",
    "edges",
    "mean_strength"
]].copy()


# Add run + type columns to match random_network_df
r11_structural["run"] = -1
r11_structural["type"] = r11_structural["phase"].apply(
    lambda x: f"observed_{x}"
)

r11_structural

In [ ]:
combined_df = pd.concat(
    [yhat_structural, random_structural, r11_structural],
    ignore_index=True
)

In [ ]:
combined_df

In [ ]:
combined_df_network_path = os.path.join(
    survivor_dir,
    f"all_network_metrics_Oct2021_{revision}_behavior_count.csv"
)

combined_df.to_csv(combined_df_network_path, index = False)

# Plot

In [ ]:
combined_df_network_path = os.path.join(
    survivor_dir,
    f"all_network_metrics_Oct2021_{revision}_behavior.csv"
)

combined_df = pd.read_csv(combined_df_network_path)

In [ ]:
plot_metrics = ["nodes", "edges", "mean_strength"]
 
palette = {
    "observed_pre": "seagreen",
    "observed_post": "salmon",
    "random_post": "indianred",
    "yhat_post": "darkred"
}

fig, axes = plt.subplots(
    1, len(plot_metrics),
    figsize=(10, 4),  
    sharey=False
)

# if only one metric, make axes iterable
if len(plot_metrics) == 1:
    axes = [axes]

for ax, metric in zip(axes, plot_metrics):

    # ----------------------------
    # 1) Random distribution
    # ----------------------------
    rand_vals = combined_df.loc[
        combined_df["month"] == "random_post",
        metric
    ].dropna()

    rand_mean = rand_vals.mean()
    rand_std = rand_vals.std()
    rand_ci = 1.96 * rand_std   # your original style (not /sqrt(n))

    # ----------------------------
    # 2) Yhat distribution
    # ----------------------------
    yhat_vals = combined_df.loc[
        combined_df["month"] == "yhat_post",
        metric
    ].dropna()

    yhat_mean = yhat_vals.mean()
    yhat_std = yhat_vals.std()
    yhat_ci = 1.96 * yhat_std   # same style for comparability

    # ----------------------------
    # 3) Observed values
    # ----------------------------
    obs_pre = combined_df.loc[
        combined_df["type"] == "observed_pre",
        metric
    ].mean()

    obs_post = combined_df.loc[
        combined_df["type"] == "observed_post",
        metric
    ].mean()

    # ----------------------------
    # 4) Plot
    # ----------------------------
    labels = ["Pre", "Post", "Random", "Yhat"]
    x = np.arange(len(labels))

    values = [obs_pre, obs_post, rand_mean, yhat_mean]
    colors = [
        palette["observed_pre"],
        palette["observed_post"],
        palette["random_post"],
        palette["yhat_post"]
    ]

    ax.bar(x, values, color=colors, width=0.65)

    # error bars only for simulated distributions
    ax.errorbar(
        x[2], rand_mean, yerr=rand_ci,
        fmt="none", color="black", capsize=4, linewidth=1
    )
    ax.errorbar(
        x[3], yhat_mean, yerr=yhat_ci,
        fmt="none", color="black", capsize=4, linewidth=1
    )

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title(metric.replace("_", " ").title())

    for spine in ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
plt.show()

# compute centralites for controlled random

In [ ]:
def ensure_str_nodes(G):
    if any(not isinstance(n, str) for n in G.nodes()):
        return nx.relabel_nodes(G, {n: str(n) for n in G.nodes()}, copy=True)
    return G

In [ ]:
def compute_node_centralities_contact_fast(G, weight="weight", approx_betweenness_k=500, seed=0):
    # build inv_weight without mutating original edge dicts too much
    inv = {}
    for u, v, d in G.edges(data=True):
        w = d.get(weight, 0.0)
        inv[(u, v)] = (1.0 / w) if (w is not None and w > 0) else 0.0
    nx.set_edge_attributes(G, inv, "inv_weight")

    out = pd.DataFrame(index=list(G.nodes()))
    out["raw_degree"] = pd.Series(dict(G.degree()))
    out["strength"] = pd.Series(dict(G.degree(weight=weight)))

    # closeness using inv_weight as distance
    out["closeness_centrality"] = pd.Series(nx.closeness_centrality(G, distance="inv_weight"))

    # clustering (unweighted, unless you want weighted clustering separately)
    out["clustering_coefficient"] = pd.Series(nx.clustering(G))

    # # betweenness: approximate for speed
    # if approx_betweenness_k is None:
    #     out["betweenness_centrality"] = pd.Series(nx.betweenness_centrality(G))
    # else:
    #     out["betweenness_centrality"] = pd.Series(
    #         nx.betweenness_centrality(G, k=approx_betweenness_k, seed=seed)
    #     )

    return out

In [ ]:
cf_graph_files = sorted(
    [f for f in os.listdir(boot_dir_w)
     if f.startswith("CF_graph_run") and f.endswith(".pkl")]
)

cf_node_rows = []
cf_network_rows = []

In [ ]:
for f in cf_graph_files:

    run = int(f.split("run")[1].split("_")[0])
    graph_path = os.path.join(boot_dir_w, f)

    with open(graph_path, "rb") as fp:
        G = pickle.load(fp)

    nodes = G.number_of_nodes()
    edges = G.number_of_edges()

    print(f"\n🔁 Run {run:04d}")
    print(f"   Nodes: {nodes}")
    print(f"   Edges: {edges}")

    if nodes == 0:
        print("   ⚠️ Empty graph — skipping.")
        continue

    # Ensure consistent node type
    G = ensure_str_nodes(G)

    # -----------------------------
    # Node-level centralities
    # -----------------------------
    node_cent = compute_node_centralities_contact_fast(
        G,
        weight="weight",
        approx_betweenness_k=500,
        seed=0
    )

    print("   ✅ Node centralities computed")

    # Quick sanity stats
    print(f"      Mean strength: {node_cent['strength'].mean():.3f}")
    print(f"      Mean clustering: {node_cent['clustering_coefficient'].mean():.3f}")

    node_cent = node_cent.reset_index().rename(columns={"index": "node"})
    node_cent["run"] = run
    node_cent["phase"] = "yhat_post"
    node_cent["revision"] = revision

    cf_node_rows.append(node_cent)

    # -----------------------------
    # Network-level summary
    # -----------------------------
    row = {
        "run": run,
        "revision": revision,
        "phase": "yhat_post",
        "nodes": nodes,
        "edges": edges,

        "mean_strength": float(node_cent["strength"].mean()),
        "median_strength": float(node_cent["strength"].median()),
        "mean_closeness": float(node_cent["closeness_centrality"].mean()),
        "median_closeness": float(node_cent["closeness_centrality"].median()),
        # "mean_betweenness": float(node_cent["betweenness_centrality"].mean()),
        # "median_betweenness": float(node_cent["betweenness_centrality"].median()),
        "mean_clustering": float(node_cent["clustering_coefficient"].mean()),
        "median_clustering": float(node_cent["clustering_coefficient"].median()),
    }

    cf_network_rows.append(row)

    print("   ✅ Network metrics aggregated")

In [ ]:
cf_nodes_df = pd.concat(cf_node_rows, ignore_index=True) if len(cf_node_rows) else pd.DataFrame()
cf_network_df = pd.DataFrame(cf_network_rows) if len(cf_network_rows) else pd.DataFrame()

nodes_csv = os.path.join(survivor_dir, f"cf_node_metrics_{revision}_yhat.csv")
net_csv   = os.path.join(survivor_dir, f"cf_network_metrics_{revision}_yhat.csv")

cf_nodes_df.to_csv(nodes_csv, index=False)
cf_network_df.to_csv(net_csv, index=False)

print("Saved:")
print(" -", nodes_csv)
print(" -", net_csv)

In [ ]:
# merge with other centralities

old_combined_path = os.path.join(survivor_dir, f"combined_node_centralities_{revision}.csv")
combined_node_df = pd.read_csv(old_combined_path)

yhat_nodes_path = os.path.join(survivor_dir, f"cf_node_metrics_{revision}_yhat.csv")
yhat_node_df = pd.read_csv(yhat_nodes_path)

# ensure phase label is exactly what you want downstream
# (use one consistent label everywhere)
yhat_node_df["phase"] = "yhat_post"   

# If old combined has a 'node' column but new file maybe stores as string/object, this helps consistency
if "node" in combined_node_df.columns and "node" in yhat_node_df.columns:
    combined_node_df["node"] = combined_node_df["node"].astype(str)
    yhat_node_df["node"] = yhat_node_df["node"].astype(str)

# Union of columns so concat doesn't silently drop anything
all_cols = sorted(set(combined_node_df.columns).union(set(yhat_node_df.columns)))

combined_node_df = combined_node_df.reindex(columns=all_cols)
yhat_node_df     = yhat_node_df.reindex(columns=all_cols)

combined_node_df_updated = pd.concat(
    [combined_node_df, yhat_node_df],
    ignore_index=True
)

combined_save_path = os.path.join(survivor_dir, f"combined_node_centralities_{revision}_with_yhat.csv")

combined_node_df_updated.to_csv(combined_save_path, index=False)

print("Saved merged combined node centralities:")
print(" -", combined_save_path)
print("Rows before:", len(combined_node_df))
print("Rows added (yhat):", len(yhat_node_df))
print("Rows after:", len(combined_node_df_updated))
print("Phase counts:")
print(combined_node_df_updated["phase"].value_counts(dropna=False))

# Plotting Edge weights:

In [ ]:
combined_save_path = os.path.join(survivor_dir, f"combined_node_centralities_{revision}_with_yhat.csv")
df = pd.read_csv(combined_save_path)
df.head()

In [ ]:
df["phase"].unique()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))

phase_map = {
    "pre": "Pre-Disaster",
    "post": "Post-Disaster",
    "random_post": "Random Counterfactual",
    "yhat_post": "Behavior-informed Counterfactual"
}

phase_color = {
    "pre": "seagreen",
    "post": "orangered",
    "random_post": "indianred",
    "yhat_post": "darkred"
}

phase_marker = {
    "pre": "o",
    "post": "s",
    "random_post": "x",
    "yhat_post": "v"
}

all_strength = df.loc[df["strength"] > 0, "strength"]
bins = np.logspace(np.log10(all_strength.min()), np.log10(all_strength.max()), 25)
bin_centers = np.sqrt(bins[:-1] * bins[1:])

for phase in ["pre", "post"]:
    df_phase = df[df["phase"] == phase]
    strengths = pd.to_numeric(df_phase["strength"], errors="coerce")
    strengths = strengths[strengths > 0]

    counts, _ = np.histogram(strengths, bins=bins, density=True)
    counts = counts.astype(float)
    counts[counts == 0] = np.nan

    ax.plot(
        bin_centers,
        counts,
        marker=phase_marker[phase],
        color=phase_color[phase],
        linewidth=2
    )

    mean_val = strengths.mean()
    if pd.notna(mean_val) and mean_val > 0:
        ax.axvline(mean_val, color=phase_color[phase], linestyle="--", linewidth=1.5)

random_df = df[df["phase"] == "random_post"]
run_distributions = []

for run in random_df["run"].dropna().unique():
    df_run = random_df[random_df["run"] == run]
    strengths = pd.to_numeric(df_run["strength"], errors="coerce")
    strengths = strengths[strengths > 0]

    counts, _ = np.histogram(strengths, bins=bins, density=True)
    run_distributions.append(counts)

run_distributions = np.array(run_distributions, dtype=float)
mean_random = np.nanmean(run_distributions, axis=0)
std_random = np.nanstd(run_distributions, axis=0)
mean_random[mean_random == 0] = np.nan

ax.plot(
    bin_centers,
    mean_random,
    color=phase_color["random_post"],
    marker=phase_marker["random_post"],
    linewidth=2
)

lower_r = np.maximum(mean_random - std_random, 1e-12)
upper_r = mean_random + std_random

ax.fill_between(
    bin_centers,
    lower_r,
    upper_r,
    color=phase_color["random_post"],
    alpha=0.25
)

random_strength_vals = pd.to_numeric(random_df["strength"], errors="coerce")
random_strength_vals = random_strength_vals[random_strength_vals > 0]
mean_val_random = random_strength_vals.mean()
if pd.notna(mean_val_random) and mean_val_random > 0:
    ax.axvline(mean_val_random, color=phase_color["random_post"], linestyle="--", linewidth=1.5)

yhat_df = df[df["phase"] == "yhat_post"]
yhat_run_distributions = []

for run in yhat_df["run"].dropna().unique():
    df_run = yhat_df[yhat_df["run"] == run]
    strengths = pd.to_numeric(df_run["strength"], errors="coerce")
    strengths = strengths[strengths > 0]

    counts, _ = np.histogram(strengths, bins=bins, density=True)
    yhat_run_distributions.append(counts)

yhat_run_distributions = np.array(yhat_run_distributions, dtype=float)
mean_yhat = np.nanmean(yhat_run_distributions, axis=0)
std_yhat = np.nanstd(yhat_run_distributions, axis=0)
mean_yhat[mean_yhat == 0] = np.nan

ax.plot(
    bin_centers,
    mean_yhat,
    color=phase_color["yhat_post"],
    marker=phase_marker["yhat_post"],
    linewidth=2
)

lower_y = np.maximum(mean_yhat - std_yhat, 1e-12)
upper_y = mean_yhat + std_yhat

ax.fill_between(
    bin_centers,
    lower_y,
    upper_y,
    color=phase_color["yhat_post"],
    alpha=0.20
)

yhat_strength_vals = pd.to_numeric(yhat_df["strength"], errors="coerce")
yhat_strength_vals = yhat_strength_vals[yhat_strength_vals > 0]
mean_val_yhat = yhat_strength_vals.mean()
if pd.notna(mean_val_yhat) and mean_val_yhat > 0:
    ax.axvline(mean_val_yhat, color=phase_color["yhat_post"], linestyle="--", linewidth=1.5)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(1e-8, 1e-0)
# increase y-tick font size
ax.tick_params(axis="y", labelsize=16)
ax.tick_params(axis="x", labelsize=16)
# ax.set_xlabel("Weighted Degree (Strength)", fontsize=16)
# ax.set_ylabel("P(k)", fontsize=18)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# plotting centralities

In [ ]:
df_plot = df.copy()

if "Condition" not in df_plot.columns:
    phase_to_condition = {
        "pre": "Pre-Disaster",
        "post": "Post-Disaster",
        "random_post": "Random Survival CF",
        "yhat_post": "Yhat Survival CF"
    }
    df_plot["Condition"] = df_plot["phase"].map(phase_to_condition)

metrics = [
    "raw_degree",
    "strength",
    "closeness_centrality",
    "clustering_coefficient"
]

metric_map = {
    "raw_degree": "Raw Degree",
    "strength": "Strength (Weighted Degree)",
    "closeness_centrality": "Closeness Centrality",
    "clustering_coefficient": "Clustering Coefficient"
}

order = [
    "Pre-Disaster",
    "Post-Disaster",
    "Random Survival CF",
    "Yhat Survival CF"
]

palette = {
    "Pre-Disaster": "seagreen",
    "Post-Disaster": "orangered",
    "Random Survival CF": "indianred",
    "Yhat Survival CF": "darkred"
}


for revision in sorted(df_plot["revision"].dropna().astype(str).unique()):
    df_rev = df_plot[df_plot["revision"].astype(str) == str(revision)].copy()

    for m in metrics:
        if m not in df_rev.columns:
            continue

        plot_df = df_rev[["Condition", m]].copy()
        plot_df[m] = pd.to_numeric(plot_df[m], errors="coerce")
        plot_df = plot_df.dropna()

        if plot_df.empty:
            continue

        plot_df = plot_df[plot_df["Condition"].isin(order)]
        if plot_df.empty:
            continue

        means = plot_df.groupby("Condition")[m].mean()

        plt.figure(figsize=(4, 3))

        ax = sns.violinplot(
            data=plot_df,
            x="Condition",
            y=m,
            order=order,
            palette=palette,
            inner="box",
            cut=0,
            linewidth=0.1
        )

        for xpos, cond in enumerate(order):
            if cond in means.index and pd.notna(means.loc[cond]):
                ax.scatter(xpos, means.loc[cond], color="black", s=10, zorder=10, alpha = 0.8)

        title_metric = metric_map.get(m, m)

        pre_mean = means.get("Pre-Disaster", np.nan)
        post_mean = means.get("Post-Disaster (Remaining Pre-Users)", np.nan)
        rand_mean = means.get("Random Survival CF", np.nan)
        yhat_mean = means.get("Yhat Survival CF", np.nan)

        ax.set_xticklabels(["", "", "", ""])
        ax.set_xlabel("")
        ax.set_ylabel("")

        # increase y-tick font size
        ax.tick_params(axis="y", labelsize=16)

        for spine in ax.spines.values():
            spine.set_visible(False)

        plt.tight_layout()
        plt.show()